# Epic Game Pass When? - pipeline runner

Run top to bottom to refresh the app. Refreshes are **quarterly**; the site shows the collection date and when the next update is due.

**Automatic:** Game Pass and PS Plus data (downloaded from the public sheets in step 0).

**By hand, before running:**
1. **Epic** - add the new weeks at the top of `data/raw/NEW_Epic Games List from PCGamer.txt`, under the `Epic Store free games in <year>` header, newest first. One date range per line, games on the lines below it.
2. **Humble** - add each new month at the very top of `data/raw/NEW_Humble Bundle Up to December 2025 Games.txt`, as `<Month> <Year> Games` followed by one title per line. The file must keep its full history.

Nothing ships unless the backtest gate (step 4) and pre-flight (step 6) both pass.

In [ ]:
from pipeline import config, fetch, ingest, enrich, train, deploy, preflight

print("Repo root:", config.REPO_ROOT)
print("RAWG keys found:", len(config.rawg_keys()))

## 0. Fetch - Game Pass + PS Plus sheets -> `data/raw/`
Downloads both catalogue sheets and converts them for ingest. Stops with an error if either sheet's column layout has changed, rather than reading the wrong columns.

In [ ]:
fetch.run()

## 1. Ingest - raw files -> `data/processed/`

In [ ]:
ingest.run()

## 2. Enrich - RAWG fill + merge -> `data/canonical/`
Only games not already known cost an API call.

In [ ]:
enrich.run()

## 2a. Epic history - add any giveaways our list is missing
Merges the open-source Epic promotion record (repeats included) into `Epic.csv`. Safe to rerun: it only adds what is missing.

In [ ]:
from pipeline import epic_history
epic_history.run()

## 3. Train - `data/canonical/` -> `models/`
One calibrated quantile model per service. The in-sample error printed here is a did-it-run check, not accuracy.

In [ ]:
import pandas as pd
metrics = train.run()
pd.DataFrame(metrics)

## 4. Backtest gate
Each model must beat the naive baseline on time-ordered data. **Do not continue if any platform fails.**

In [ ]:
from pipeline import backtest
results = backtest.run()
report = pd.DataFrame(results)
failing = [r['platform'] for r in results if r.get('beats_baseline') is False]
if failing:
    print('WARNING: these models do not beat the best baseline (Phase 5 target):', failing)
report

## 5. Deploy - sync into `apps/backend/`
Copies data and models together, and regenerates the arrival-chance table and the data-status file the site displays.

In [ ]:
deploy.run()

## 5b. Precompute - answers the site serves without calling the backend
Fetches RAWG details for any new games (only new ones cost a request), then runs every game through the deployed backend. Pre-flight refuses to pass if these answers were made from a different model, dataset or backend code, so run this after every deploy.

In [ ]:
from pipeline import popularity, rawg_details, precompute
popularity.run()      # locates new games on RAWG
rawg_details.run()    # the details the site's requests are built from
precompute.run()

## 6. Pre-flight
Exercises the deployed backend in-process. **Must pass before pushing.**

In [ ]:
assert preflight.run() == 0, "Pre-flight failed - do not push"

## 7. Review, then ship to `dev`
Pushing `dev` redeploys the dev site. Production is always a separate manual merge.

In [ ]:
!git status --short

In [ ]:
# Uncomment to ship to dev (auto-deploys dev; prod is a separate manual merge):
# !git add -A && git commit -m "data refresh: retrain models + redeploy" && git push origin dev